# NMF on ModulAir 00686

In [1]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [2]:
#importing data from Modulair MOD-00068
data = pd.read_csv('MOD-00679-2-27data.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-31T23:59:42Z,577612018,2025-12-31T18:59:42Z,MOD-00679,49.4,0.2,5.711,0.844,0.341,0.071,0.046,...,31.630,27.138,14310.0,14311.0,14312.0,14466.0,14491.0,14541.0,14516.0,1.58
2025-12-31T23:58:42Z,577612014,2025-12-31T18:58:42Z,MOD-00679,49.2,0.1,5.916,0.818,0.337,0.107,0.059,...,31.394,27.506,14310.0,14311.0,14312.0,14466.0,14491.0,14541.0,14516.0,1.20
2025-12-31T23:57:42Z,577610060,2025-12-31T18:57:42Z,MOD-00679,49.0,0.1,5.734,1.020,0.294,0.103,0.056,...,31.165,27.878,14310.0,14311.0,14312.0,14466.0,14491.0,14541.0,14516.0,1.05
2025-12-31T23:56:42Z,577610059,2025-12-31T18:56:42Z,MOD-00679,48.9,0.1,5.723,0.806,0.241,0.078,0.036,...,30.933,28.238,14310.0,14311.0,14312.0,14466.0,14491.0,14541.0,14516.0,2.02
2025-12-31T23:55:42Z,577610061,2025-12-31T18:55:42Z,MOD-00679,49.0,0.1,5.317,0.855,0.325,0.087,0.033,...,30.928,27.878,14310.0,14311.0,14312.0,14466.0,14491.0,14541.0,14516.0,1.31


In [3]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
timestamp,,,,,,,,,,,
2025-12-31T23:59:42Z,2025-12-31T18:59:42Z,813.531,2.469,31.630,27.138,5.711,0.844,0.341,0.071,0.046,0.025
2025-12-31T23:58:42Z,2025-12-31T18:58:42Z,806.176,2.584,31.394,27.506,5.916,0.818,0.337,0.107,0.059,0.019
2025-12-31T23:57:42Z,2025-12-31T18:57:42Z,774.709,2.584,31.165,27.878,5.734,1.020,0.294,0.103,0.056,0.022
2025-12-31T23:56:42Z,2025-12-31T18:56:42Z,758.770,2.584,30.933,28.238,5.723,0.806,0.241,0.078,0.036,0.011
2025-12-31T23:55:42Z,2025-12-31T18:55:42Z,760.341,2.584,30.928,27.878,5.317,0.855,0.325,0.087,0.033,0.011


In [4]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31T18:59:42Z,813.531,2.469,31.630,27.138,5.711,0.844,0.341,0.071,0.046,0.025
1,2025-12-31T18:58:42Z,806.176,2.584,31.394,27.506,5.916,0.818,0.337,0.107,0.059,0.019
2,2025-12-31T18:57:42Z,774.709,2.584,31.165,27.878,5.734,1.020,0.294,0.103,0.056,0.022
3,2025-12-31T18:56:42Z,758.770,2.584,30.933,28.238,5.723,0.806,0.241,0.078,0.036,0.011
4,2025-12-31T18:55:42Z,760.341,2.584,30.928,27.878,5.317,0.855,0.325,0.087,0.033,0.011


In [5]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'],
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31 18:59:42,813.531,2.469,31.630,27.138,5.711,0.844,0.341,0.071,0.046,0.025
1,2025-12-31 18:58:42,806.176,2.584,31.394,27.506,5.916,0.818,0.337,0.107,0.059,0.019
2,2025-12-31 18:57:42,774.709,2.584,31.165,27.878,5.734,1.020,0.294,0.103,0.056,0.022
3,2025-12-31 18:56:42,758.770,2.584,30.933,28.238,5.723,0.806,0.241,0.078,0.036,0.011
4,2025-12-31 18:55:42,760.341,2.584,30.928,27.878,5.317,0.855,0.325,0.087,0.033,0.011


In [6]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
4,2025-04-25 13:00:00,1224.45,6.94,72.88,5.97,7.20,0.84,0.35,0.11,0.14,0.09
5,2025-04-25 14:00:00,1108.79,6.83,70.61,16.71,6.88,0.91,0.41,0.13,0.16,0.10
6,2025-04-25 15:00:00,986.69,9.03,68.21,6.06,6.16,0.91,0.41,0.13,0.16,0.10
7,2025-04-25 16:00:00,935.48,11.95,64.77,4.35,4.90,0.77,0.39,0.14,0.17,0.10
8,2025-04-25 17:00:00,880.92,15.14,63.88,4.01,5.84,1.24,0.46,0.14,0.17,0.09
...,...,...,...,...,...,...,...,...,...,...,...
5962,2025-12-31 14:00:00,693.96,29.28,34.29,1.96,5.77,0.55,0.17,0.05,0.03,0.01
5963,2025-12-31 15:00:00,699.64,29.91,33.66,1.83,5.25,0.59,0.17,0.04,0.04,0.01
5964,2025-12-31 16:00:00,726.75,29.96,32.84,2.35,5.82,0.63,0.20,0.05,0.04,0.02
5965,2025-12-31 17:00:00,769.36,30.97,29.39,2.45,6.48,0.79,0.24,0.07,0.05,0.02


In [7]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-04-25 13:00:00,1224.45,6.94,72.88,5.97,7.20,0.84,0.35,0.11,0.14,0.09
2025-04-25 14:00:00,1108.79,6.83,70.61,16.71,6.88,0.91,0.41,0.13,0.16,0.10
2025-04-25 15:00:00,986.69,9.03,68.21,6.06,6.16,0.91,0.41,0.13,0.16,0.10
2025-04-25 16:00:00,935.48,11.95,64.77,4.35,4.90,0.77,0.39,0.14,0.17,0.10
2025-04-25 17:00:00,880.92,15.14,63.88,4.01,5.84,1.24,0.46,0.14,0.17,0.09


In [8]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [9]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled

# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [10]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-04-25 13:00:00,0.536301,0.147252,0.833295,0.082859,0.087923,0.031390,0.025307,0.025229,0.040000,0.065217
2025-04-25 14:00:00,0.485643,0.144918,0.807340,0.231922,0.084015,0.034006,0.029646,0.029817,0.045714,0.072464
2025-04-25 15:00:00,0.432164,0.191598,0.779899,0.084108,0.075223,0.034006,0.029646,0.029817,0.045714,0.072464
2025-04-25 16:00:00,0.409734,0.253554,0.740567,0.060375,0.059836,0.028774,0.028200,0.032110,0.048571,0.072464
2025-04-25 17:00:00,0.385837,0.321239,0.730391,0.055656,0.071315,0.046338,0.033261,0.032110,0.048571,0.065217


In [11]:
data.to_csv('MOD-000679_timeseries_hourly_scaled.csv')

## Implementing NMF

In [12]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [13]:
W = nmf.fit_transform(data)
H = nmf.components_
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-04-25 13:00:00,0.394339,0.188586,0.881458,0.043195,0.126398,0.038118,0.027657,0.025608,0.046114,0.088293
2025-04-25 14:00:00,0.379546,0.179527,0.849690,0.041666,0.115486,0.038218,0.030426,0.029429,0.051047,0.093618
2025-04-25 15:00:00,0.367016,0.211270,0.802294,0.040172,0.096874,0.031360,0.027582,0.027791,0.049009,0.089562
2025-04-25 16:00:00,0.362136,0.267711,0.756180,0.039336,0.076452,0.025630,0.026341,0.028002,0.049391,0.088525
2025-04-25 17:00:00,0.372914,0.325639,0.733240,0.039782,0.084763,0.030742,0.027339,0.027661,0.047312,0.084418
...,...,...,...,...,...,...,...,...,...,...
2025-12-31 14:00:00,0.323565,0.615738,0.384507,0.030941,0.065985,0.021116,0.007555,0.003374,0.006525,0.018160
2025-12-31 15:00:00,0.324045,0.629672,0.377816,0.030929,0.061738,0.019552,0.006995,0.003124,0.006301,0.017666
2025-12-31 16:00:00,0.326236,0.633557,0.372491,0.030949,0.069231,0.025291,0.010153,0.005578,0.008633,0.020134


In [14]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.018333,0.008988,0.019904,0.192789
1,0.017326,0.006995,0.023784,0.186278
2,0.020806,0.003784,0.023231,0.173180
3,0.026801,0.000252,0.024380,0.158699
4,0.032730,0.003485,0.023199,0.148881
...,...,...,...,...
5866,0.063563,0.010785,0.000000,0.045232
5867,0.065051,0.009986,0.000000,0.042742
5868,0.065293,0.012092,0.001568,0.041077
5869,0.067789,0.016085,0.003647,0.031124


In [15]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [16]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,3.727858,9.608939,2.923775,0.342579,0.038432,0.000000,0.000000,0.000000,0.030446,0.101351
Feature 2,1.454297,0.449299,0.510910,0.082000,3.924587,1.957884,0.700497,0.312801,0.082317,0.000000
Feature 3,0.533640,0.395855,0.000000,0.044224,0.000000,1.030997,1.073206,1.145346,1.458795,1.833238
Feature 4,1.568048,0.002614,4.270289,0.183085,0.469012,0.000000,0.000000,0.000000,0.081852,0.259073


In [17]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-04-25 13:00:00,0.018333,0.008988,0.019904,0.192789
2025-04-25 14:00:00,0.017326,0.006995,0.023784,0.186278
2025-04-25 15:00:00,0.020806,0.003784,0.023231,0.173180
2025-04-25 16:00:00,0.026801,0.000252,0.024380,0.158699
2025-04-25 17:00:00,0.032730,0.003485,0.023199,0.148881
...,...,...,...,...
2025-12-31 14:00:00,0.063563,0.010785,0.000000,0.045232
2025-12-31 15:00:00,0.065051,0.009986,0.000000,0.042742
2025-12-31 16:00:00,0.065293,0.012092,0.001568,0.041077


In [18]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,3.727858,9.608939,2.923775,0.342579,0.038432,0.000000,0.000000,0.000000,0.030446,0.101351
Factor 2,1.454297,0.449299,0.510910,0.082000,3.924587,1.957884,0.700497,0.312801,0.082317,0.000000
Factor 3,0.533640,0.395855,0.000000,0.044224,0.000000,1.030997,1.073206,1.145346,1.458795,1.833238
Factor 4,1.568048,0.002614,4.270289,0.183085,0.469012,0.000000,0.000000,0.000000,0.081852,0.259073


In [19]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.548439,0.103401,0.028498,0.298511,0.021152
no,0.527478,0.061018,0.024717,0.364778,0.022009
no2,0.967458,0.021862,0.014467,0.000341,-0.004128
o3,0.338161,0.028558,0.000000,0.639100,-0.005819
bin0,0.015348,0.757453,0.000000,0.242368,-0.015169
bin1,0.000000,0.795658,0.314690,0.000000,-0.110348
bin2,0.000000,0.509958,0.586811,0.000000,-0.096769
bin3,0.000000,0.272692,0.749940,0.000000,-0.022632
bin4,0.043533,0.056883,0.757134,0.151444,-0.008994
bin5,0.092330,0.000000,0.606213,0.305401,-0.003944


In [20]:
results.to_csv('679_factor_results.csv')
comp.to_csv('679_factor_comp.csv')
res.to_csv('679_factor_resid.csv')